# RF Cavity Neural Operator — Colab

Tek notebook ile uçtan uca: **veri üretimi (gmsh + P2 FEM) → PKL dönüşümü → eğitim → inference → görseller**.

1. Aşağıdaki **Ayarlar** hücresini düzenleyin (`MODE="smoke"` ile ~birkaç dakikada tüm akışı dener, `MODE="full"` gerçek eğitimdir; GPU önerilir: *Runtime → Change runtime type → GPU*).
2. *Runtime → Run all*.

Varsayılan model `configs/spectral_no.yaml` (SpectralNO): P1 eleman montajı (gerçek Rayleigh–Ritz üst sınırı), fiziksel frekans `f = c·√λ/(2π·scale)`, alan-ağırlıklı kayıp. Ayrıntılar: `docs/14`–`docs/17`.

In [ ]:
# ── Ayarlar ─────────────────────────────────────────────────────────
REPO_URL = "https://github.com/KorayGokceler/rf_cavity_neural_operator.git"
BRANCH = "claude/apply-math-findings"
MODE = "smoke"                          # "smoke" (hızlı deneme) | "full" (gerçek eğitim)
MODEL_CONFIG = "configs/spectral_no.yaml"   # SpectralNO; GNOT için "configs/default.yaml"
USE_DRIVE = False                       # True → veri/log/sonuçlar Google Drive'a yazılır (oturum kapansa da kalır)

if MODE == "smoke":
    N_TOTAL, EPOCHS = 20, 5
    EXTRA = "training.batch_size=4 model.embed_dim=32 training.check_val_every_n_epoch=1"
else:
    N_TOTAL, EPOCHS = 5000, 300
    EXTRA = ""                          # örn. "training.batch_size=8" (GPU belleği yetmezse)

In [ ]:
# ── Repo ────────────────────────────────────────────────────────────
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.exists('/content/rf_cavity_neural_operator'):
    !git clone -q -b {BRANCH} {REPO_URL} /content/rf_cavity_neural_operator
%cd /content/rf_cavity_neural_operator
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!git log --oneline -1

In [ ]:
# ── Kurulum (gmsh sistem kütüphaneleri + Python paketleri) ────────────
!apt-get -qq install -y libglu1-mesa libxrender1 libxcursor1 libxft2 libxinerama1 > /dev/null
!pip -q install -r requirements.txt
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "yok (CPU)")

In [ ]:
# ── Yollar ──────────────────────────────────────────────────────────
WORK = "/content/drive/MyDrive/rf_cavity" if USE_DRIVE else "/content/rf_work"
H5   = f"{WORK}/rf_cavity_{N_TOTAL}.h5"
PKL  = f"{WORK}/dataset_{N_TOTAL}.pkl"
LOGS = f"{WORK}/training_logs"
EXP  = f"{os.path.basename(MODEL_CONFIG).replace('.yaml', '')}_{MODE}"
OUT  = f"{WORK}/inference_{EXP}"
os.makedirs(WORK, exist_ok=True)
print(H5, PKL, LOGS, EXP, sep="\n")

In [ ]:
# ── 1. Veri üretimi (varsa atlanır) ──────────────────────────────────
# 6 mod saklanır (λ3–λ4 aralığı dar olabiliyor); eğitim config'teki mode_indices'i kullanır.
if not os.path.exists(H5):
    !python src/data_gen/dataset_generator.py --mode random --n_total {N_TOTAL} --n_eigen_modes 6 --seed 0 --n_plot 5 --h5_filename {H5} --plot_dir {WORK}/dataset_plots
else:
    print("Mevcut:", H5)

In [ ]:
# ── 2. H5 → PKL (12 feature + ölçek + üçgenler) ─────────────────────
if not os.path.exists(PKL):
    !python convert.py --h5_filepath {H5} --output_path {PKL} --modes 0 1 2
else:
    print("Mevcut:", PKL)

In [ ]:
# ── 3. Eğitim ───────────────────────────────────────────────────────
!python train.py --config {MODEL_CONFIG} --override dataset.data_path={PKL} training.max_epochs={EPOCHS} training.num_workers=2 training.log_dir={LOGS} training.exp_name={EXP} {EXTRA}

In [ ]:
# ── 4. Inference (test split, en iyi checkpoint) ─────────────────────
!python infer.py --checkpoint {LOGS}/{EXP} --data_path {PKL} --split test --num_samples 5 --output_dir {OUT}

In [ ]:
# ── 5. Sonuçlar ─────────────────────────────────────────────────────
import glob
from IPython.display import Image, display
for p in sorted(glob.glob(f"{OUT}/*.png"))[:5]:
    print(os.path.basename(p)); display(Image(filename=p))

In [ ]:
# ── (İsteğe bağlı) Eğitim eğrileri: TensorBoard ─────────────────────
%load_ext tensorboard
%tensorboard --logdir {LOGS}

In [ ]:
# ── (İsteğe bağlı) Birim testleri (~15 sn) ──────────────────────────
!python -m pytest -q